# 08. Eficiência Computacional, Energia e Pegada de Carbono

**DCT1401 — Inteligência Artificial (UFRN, 2026.2)**
**Aluna:** Mariana dos Santos Batista Medeiros (Matrícula: 20230083988)

**Atividade 02 — Avaliação Comparativa de Modelos**

Este notebook compara a eficiência computacional do Sistema Especialista, da Árvore de Decisão, da Random Forest e do Gradient Boosting, conforme a Seção 5.3 do enunciado.

A avaliação contempla tempo de treinamento, tempo de inferência, vazão, memória, tamanho dos modelos em disco, complexidade, consumo energético estimado e emissão estimada de CO₂.

Os mesmos modelos e a mesma divisão de dados utilizados nos notebooks anteriores são preservados. As medições principais serão realizadas sob condições computacionais padronizadas, com repetições e registro da variabilidade dos tempos.

O Sistema Especialista é avaliado somente quanto à inferência computacional. Seu esforço de elaboração das regras, realizado manualmente, é apresentado separadamente.

A estimativa energética utiliza a relação entre potência e tempo de execução, com fator de emissão de 45,8 gCO₂/kWh, conforme fornecido pelo professor. Os resultados são estimativas, não medições elétricas diretas.

Ao final, serão discutidas as vantagens e limitações de cada família de modelos para implantação em dispositivos com recursos limitados e em servidores.

**Nota sobre a reprodutibilidade:** Os tempos de processamento, as alocações de memória e as estimativas energéticas dependem das condições do ambiente computacional. Os resultados principais apresentados correspondem à execução de 21/10/2026, no ambiente documentado na Seção 1.1. Reexecuções podem produzir valores diferentes, motivo pelo qual as conclusões consideram também os padrões observados em outras rodadas de medição.

**Configuração do ambiente — Google Colab**

Esta etapa conecta o Google Colab ao Google Drive e configura os caminhos necessários para acessar o dataset original e os módulos Python do projeto.

A execução pressupõe que a pasta `projeto-cardiovascular-ia-02` esteja localizada em `Meu Drive`, conforme as instruções disponíveis no `README.md` da Atividade 02.

O código verifica a existência do dataset e prepara o ambiente para a execução das etapas seguintes.

In [ ]:
from google.colab import drive
from pathlib import Path

import os
import sys
import platform
import importlib

import numpy as np
import pandas as pd
import joblib

# Conectar ao Drive
drive.mount("/content/drive")

# Localizar o projeto
PROJETO = Path(
    "/content/drive/MyDrive/projeto-cardiovascular-ia-02"
)

assert PROJETO.is_dir(), "Pasta do projeto não encontrada."

os.chdir(PROJETO / "notebooks")

if str(PROJETO) not in sys.path:
    sys.path.insert(0, str(PROJETO))

# Conferir os arquivos necessários
arquivos_necessarios = [
    "data/raw/cardiovascular_disease_prediction.csv",
    "src/pipeline_data.py",
    "src/profiling.py",
    "src/expert_system.py",
    "rules/regras_classificacao.txt",
    "models/decision_tree_clf.joblib",
    "models/decision_tree_reg.joblib",
    "models/random_forest_clf.joblib",
    "models/random_forest_reg.joblib",
    "models/gradient_boosting_clf.joblib",
    "models/gradient_boosting_reg.joblib",
    "logs/cronometro_regras.csv",
]

faltando = [
    arquivo
    for arquivo in arquivos_necessarios
    if not (PROJETO / arquivo).is_file()
]

assert not faltando, f"Arquivos ausentes: {faltando}"

# Diretórios dos resultados
pasta_tabelas = PROJETO / "reports" / "tables"
pasta_figuras = PROJETO / "reports" / "figures"
pasta_logs = PROJETO / "logs"

for pasta in [pasta_tabelas, pasta_figuras, pasta_logs]:
    pasta.mkdir(parents=True, exist_ok=True)

print("AMBIENTE CONFIGURADO COM SUCESSO")
print(f"Arquivos conferidos: {len(arquivos_necessarios)}")
print(f"Sistema operacional: {platform.platform()}")
print(f"Processador informado: {platform.processor() or 'Não identificado'}")
print(f"CPUs lógicas disponíveis: {os.cpu_count()}")
print(f"Versão do Python: {platform.python_version()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
AMBIENTE CONFIGURADO COM SUCESSO
Arquivos conferidos: 12
Sistema operacional: Linux-6.6.122+-x86_64-with-glibc2.39
Processador informado: x86_64
CPUs lógicas disponíveis: 2
Versão do Python: 3.13.16


## 1. Preparação do ambiente e dos dados

In [ ]:
import src.pipeline_data as pipeline_data
import src.profiling as profiling

importlib.reload(pipeline_data)
importlib.reload(profiling)

(
    X_train,
    X_test,
    y_train_c,
    y_test_c,
    y_train_r,
    y_test_r,
    feat_cols
) = pipeline_data.carregar_e_dividir_dados_sem_leakage(
    "../data/raw/cardiovascular_disease_prediction.csv"
)

# Garantir o mesmo protocolo experimental
assert X_train.shape == (16000, 6)
assert X_test.shape == (4000, 6)

assert X_train.index.intersection(
    X_test.index
).empty

print("DADOS RECUPERADOS")
print(f"Treinamento: {X_train.shape}")
print(f"Teste: {X_test.shape}")
print(f"Variáveis: {feat_cols}")

print("\nParâmetros energéticos:")
print(
    f"Potência assumida: "
    f"{profiling.POTENCIA_PADRAO_WATTS} W"
)
print(
    f"Fator de emissão: "
    f"{profiling.FATOR_EMISSAO_BRASIL_GCO2_KWH} gCO₂/kWh"
)

DADOS RECUPERADOS
Treinamento: (16000, 6)
Teste: (4000, 6)
Variáveis: ['Diabetes', 'Hypertension', 'Chest_Pain', 'ECG_Result', 'Blood_Glucose', 'Blood_Pressure_Systolic']

Parâmetros energéticos:
Potência assumida: 45.0 W
Fator de emissão: 45.8 gCO₂/kWh


### 1.1 Ambiente computacional

As especificações do ambiente de execução são registradas para documentar as condições das medições.

São identificados o modelo do processador, a quantidade de CPUs lógicas, a memória RAM disponibilizada, o sistema operacional e as versões das principais bibliotecas.

Como o Google Colab utiliza infraestrutura remota, os recursos disponibilizados podem variar entre sessões. Portanto, essas especificações devem corresponder à mesma sessão utilizada para medir os tempos de treinamento e inferência.

In [ ]:
import platform
import os
import sys
import psutil
import sklearn
import numpy
import pandas
import matplotlib
import joblib

from datetime import datetime, timezone

# Identificar o processador no ambiente Linux do Colab
cpu = platform.processor() or "Não identificado"

try:
    with open("/proc/cpuinfo", encoding="utf-8") as arquivo:
        for linha in arquivo:
            if linha.startswith("model name"):
                cpu = linha.split(":", 1)[1].strip()
                break
except OSError:
    pass

# RAM total disponibilizada para a máquina virtual
ram_gb = psutil.virtual_memory().total / (1024 ** 3)

ambiente = {
    "Data UTC": datetime.now(timezone.utc).isoformat(),
    "Ambiente": "Google Colab",
    "Processador": cpu,
    "CPUs lógicas": os.cpu_count(),
    "RAM total (GB)": round(ram_gb, 2),
    "Sistema operacional": platform.platform(),
    "Python": sys.version.split()[0],
    "scikit-learn": sklearn.__version__,
    "NumPy": numpy.__version__,
    "Pandas": pandas.__version__,
    "Matplotlib": matplotlib.__version__,
    "Joblib": joblib.__version__
}

tabela_ambiente = pd.DataFrame([ambiente])

tabela_ambiente.to_csv(
    pasta_tabelas / "eficiencia_ambiente.csv",
    index=False
)

print("AMBIENTE COMPUTACIONAL")
display(tabela_ambiente.T)

print("\nEspecificações salvas com sucesso!")

AMBIENTE COMPUTACIONAL


,0
Data UTC,2026-10-22T00:56:47.836144+00:00
Ambiente,Google Colab
Processador,Intel(R) Xeon(R) CPU @ 2.20GHz
CPUs lógicas,2
RAM total (GB),12.67
Sistema operacional,Linux-6.6.122+-x86_64-with-glibc2.39
Python,3.13.16
scikit-learn,1.6.1
NumPy,2.1.3
Pandas,2.2.3



Especificações salvas com sucesso!


## 2. Esforço humano de elaboração do Sistema Especialista

O Sistema Especialista foi desenvolvido manualmente, por meio da definição de 11 regras de classificação e de sua ordem de prioridade.

Para avaliar o esforço de construção, foram registradas três sessões de elaboração e pesquisa no arquivo `logs/cronometro_regras.csv`.

A análise distingue o tempo total transcorrido, as interrupções e o tempo efetivamente dedicado à atividade.

Esse esforço humano é apresentado separadamente do tempo computacional de treinamento dos modelos de aprendizado de máquina, pois são atividades de naturezas diferentes. A inferência do Sistema Especialista será medida posteriormente, utilizando o mesmo protocolo dos demais classificadores.

In [ ]:
# Recuperar o cronômetro de elaboração das regras
cronometro_regras = pd.read_csv(
    pasta_logs / "cronometro_regras.csv"
)

colunas_necessarias = [
    "sessao",
    "tempo_relogio_min",
    "tempo_interrupcoes_min",
    "tempo_minutos",
    "regras_criadas"
]

assert set(colunas_necessarias).issubset(
    cronometro_regras.columns
)

assert len(cronometro_regras) == 3

# Calcular os totais registrados
tempo_relogio = cronometro_regras[
    "tempo_relogio_min"
].sum()

tempo_interrupcoes = cronometro_regras[
    "tempo_interrupcoes_min"
].sum()

tempo_efetivo = cronometro_regras[
    "tempo_minutos"
].sum()

total_regras = int(
    cronometro_regras["regras_criadas"].sum()
)

# Conferir a consistência do cronômetro
assert np.isclose(
    tempo_relogio - tempo_interrupcoes,
    tempo_efetivo
)

assert total_regras == 11

# Consolidar o esforço humano
tabela_esforco_humano = pd.DataFrame([{
    "Sessões": len(cronometro_regras),
    "Tempo total (min)": tempo_relogio,
    "Interrupções (min)": tempo_interrupcoes,
    "Tempo efetivo (min)": tempo_efetivo,
    "Regras elaboradas": total_regras,
    "Tempo médio por regra (min)": (
        tempo_efetivo / total_regras
    )
}])

# Salvar tabela consolidada
tabela_esforco_humano.to_csv(
    pasta_tabelas / "eficiencia_esforco_humano_regras.csv",
    index=False,
    float_format="%.16g"
)

print("REGISTRO DAS SESSÕES")
display(cronometro_regras)

print("\nESFORÇO HUMANO — SISTEMA ESPECIALISTA")
display(tabela_esforco_humano.round(2))

print("\nTabela salva com sucesso!")

REGISTRO DAS SESSÕES


,sessao,tarefa,data,horario_inicio,horario_fim,tempo_relogio_min,tempo_interrupcoes_min,tempo_minutos,regras_criadas,interrupcoes,descricao
0,1,Classificacao,2026-10-16,18:00,18:30,30,10.0,20.0,5,Pausa,Criação de regras
1,2,Classificacao,2026-10-16,18:50,19:15,25,3.0,22.0,4,Descanso,Elaboração e pesquisa
2,3,Classificacao,2026-10-16,19:20,19:40,20,2.0,18.0,2,Pausa,Pesquisa e elaboração de regras



ESFORÇO HUMANO — SISTEMA ESPECIALISTA


,Sessões,Tempo total (min),Interrupções (min),Tempo efetivo (min),Regras elaboradas,Tempo médio por regra (min)
0,3,75,15.0,60.0,11,5.45



Tabela salva com sucesso!


### 2.1 Interpretação do esforço humano

A elaboração do Sistema Especialista ocorreu em três sessões, totalizando 75 minutos de tempo registrado. Descontadas as interrupções, foram contabilizados 60 minutos efetivos de trabalho para a construção de 11 regras, correspondendo a uma média de 5,45 minutos por regra.

Esse resultado representa o esforço humano de elaboração e pesquisa registrado no projeto. Não inclui necessariamente todo o tempo necessário para manutenção, validação e futuras atualizações das regras.

O tempo humano é apresentado separadamente do treinamento computacional dos modelos de aprendizado de máquina, pois essas medidas não são diretamente equivalentes.

## 3. Tempo de treinamento dos modelos de aprendizado de máquina

O tempo de treinamento computacional é avaliado para os seis modelos de aprendizado de máquina, considerando classificação e regressão.

São realizadas cinco repetições do ajuste de cada modelo, utilizando os mesmos 16.000 registros de treinamento e os hiperparâmetros previamente selecionados.

O Pipeline completo é ajustado em cada repetição, incluindo o pré-processamento. A medição não contempla a busca de hiperparâmetros nem a validação cruzada realizada anteriormente.

Para cada modelo, são apresentados o tempo médio de treinamento, o desvio padrão e o pico de alocações de memória Python observado com `tracemalloc`.

A comparação é realizada na mesma sessão computacional, utilizando o mesmo procedimento de medição. O pico de alocação Python não representa necessariamente o consumo total de memória RAM do processo.

Os modelos originais permanecem preservados, pois o treinamento é realizado em cópias temporárias.

In [ ]:
from sklearn.base import clone
import numpy as np
import pandas as pd
import joblib

# -------- RECUPERAR OS MODELOS VALIDADOS --------

modelos_classificacao = {
    "Árvore de Decisão": joblib.load(
        PROJETO / "models/decision_tree_clf.joblib"
    ),
    "Random Forest": joblib.load(
        PROJETO / "models/random_forest_clf.joblib"
    ),
    "Gradient Boosting": joblib.load(
        PROJETO / "models/gradient_boosting_clf.joblib"
    )
}

modelos_regressao = {
    "Árvore de Decisão": joblib.load(
        PROJETO / "models/decision_tree_reg.joblib"
    ),
    "Random Forest": joblib.load(
        PROJETO / "models/random_forest_reg.joblib"
    ),
    "Gradient Boosting": joblib.load(
        PROJETO / "models/gradient_boosting_reg.joblib"
    )
}

# -------- CONFIGURAR A MEDIÇÃO --------

N_REPETICOES_TREINO = 5

experimentos_treinamento = [
    ("Classificação", nome, modelo, y_train_c)
    for nome, modelo in modelos_classificacao.items()
] + [
    ("Regressão", nome, modelo, y_train_r)
    for nome, modelo in modelos_regressao.items()
]

registros_treinamento = []
detalhes_treinamento = []

# -------- MEDIR O TEMPO DE TREINAMENTO --------

for tarefa, nome, modelo, y_treino in experimentos_treinamento:

    print(f"Medindo: {nome} — {tarefa}")

    # Aquecimento não contabilizado
    modelo_aquecimento = clone(modelo)
    modelo_aquecimento.fit(X_train, y_treino)
    del modelo_aquecimento

    tempos = []
    picos_memoria = []

    for repeticao in range(1, N_REPETICOES_TREINO + 1):

        # Criar uma cópia nova, sem modificar o modelo salvo
        modelo_temporario = clone(modelo)

        _, medidas = profiling.medir_treinamento(
            modelo_temporario.fit,
            X_train,
            y_treino
        )

        tempo_s = medidas["tempo_treino_s"]
        memoria_kb = medidas["pico_memoria_treino_kb"]

        tempos.append(tempo_s)
        picos_memoria.append(memoria_kb)

        detalhes_treinamento.append({
            "Tarefa": tarefa,
            "Modelo": nome,
            "Repetição": repeticao,
            "Tempo (s)": tempo_s,
            "Pico alocação Python (KB)": memoria_kb
        })

        del modelo_temporario

    tempo_medio = float(np.mean(tempos))

    registros_treinamento.append({
        "Tarefa": tarefa,
        "Modelo": nome,
        "Repetições": N_REPETICOES_TREINO,
        "Tempo médio (s)": tempo_medio,
        "Desvio padrão (s)": float(np.std(tempos, ddof=1)),
        "Pico alocação Python (KB)": float(np.max(picos_memoria))
    })

# -------- ORGANIZAR E VALIDAR RESULTADOS --------

tabela_treinamento = pd.DataFrame(registros_treinamento)
tabela_treinamento_detalhes = pd.DataFrame(
    detalhes_treinamento
)

assert len(tabela_treinamento) == 6
assert len(tabela_treinamento_detalhes) == 30

assert np.isfinite(
    tabela_treinamento.select_dtypes(
        include="number"
    ).to_numpy()
).all()

# -------- SALVAR RESULTADOS --------

tabela_treinamento.to_csv(
    pasta_tabelas / "eficiencia_treinamento.csv",
    index=False,
    float_format="%.16g"
)

tabela_treinamento_detalhes.to_csv(
    pasta_logs / "medicoes_treinamento_final.csv",
    index=False,
    float_format="%.16g"
)

print("\nEFICIÊNCIA — TREINAMENTO COMPUTACIONAL")

display(
    tabela_treinamento.round(4)
)

print("\nMedições de treinamento salvas com sucesso!")

Medindo: Árvore de Decisão — Classificação
Medindo: Random Forest — Classificação
Medindo: Gradient Boosting — Classificação
Medindo: Árvore de Decisão — Regressão
Medindo: Random Forest — Regressão
Medindo: Gradient Boosting — Regressão

EFICIÊNCIA — TREINAMENTO COMPUTACIONAL


,Tarefa,Modelo,Repetições,Tempo médio (s),Desvio padrão (s),Pico alocação Python (KB)
0,Classificação,Árvore de Decisão,5,0.1082,0.0028,2479.4863
1,Classificação,Random Forest,5,1.3582,0.3088,2525.7119
2,Classificação,Gradient Boosting,5,0.8010,0.0069,2599.7549
3,Regressão,Árvore de Decisão,5,0.0667,0.0085,2479.5801
4,Regressão,Random Forest,5,2.6294,0.6135,2480.5215
5,Regressão,Gradient Boosting,5,1.3116,0.2389,2479.4180



Medições de treinamento salvas com sucesso!


### 3.1 Interpretação dos tempos de treinamento

As Árvores de Decisão apresentaram os menores tempos médios de treinamento nas duas tarefas. A Random Forest apresentou os maiores tempos, enquanto o Gradient Boosting apresentou valores intermediários, conforme a tabela anterior.

Os resultados são compatíveis com a maior quantidade de estimadores e operações executadas pelos ensembles, embora os tempos também dependam dos hiperparâmetros, da implementação e das condições do ambiente computacional.

Os desvios padrão indicam maior variabilidade nas medições da Random Forest, principalmente na regressão. As Árvores de Decisão apresentaram os menores desvios padrão nesta execução.

Os picos de alocação rastreados com `tracemalloc` ficaram próximos entre os seis modelos. Essa semelhança pode refletir o custo comum de pré-processamento e manipulação dos dados, enquanto parte das alocações realizadas pelas bibliotecas nativas não é capturada pelo rastreamento.

Portanto, esses valores não representam o consumo total de RAM e não são suficientemente discriminativos para comparar a memória efetivamente utilizada pelos algoritmos. O tamanho dos modelos em disco será utilizado posteriormente como indicador complementar de armazenamento.

As medições incluem o ajuste do Pipeline completo, com hiperparâmetros fixos, mas não contemplam a busca de hiperparâmetros ou a validação cruzada.

### 3.2 Custo computacional da seleção de modelos

O tempo de treinamento apresentado na Seção 3 corresponde exclusivamente ao ajuste final dos modelos com hiperparâmetros fixos.

Para complementar a análise do custo de construção, são recuperados os tempos das etapas de treinamento e seleção de hiperparâmetros realizadas nos Notebooks 04 e 05.

Nas Árvores de Decisão, o tempo registrado contempla também o treinamento das versões irrestritas. Nos ensembles, contempla a busca por validação cruzada e o ajuste dos modelos selecionados.

Esses valores não são diretamente equivalentes ao esforço humano de elaboração do Sistema Especialista, mas permitem comparar a ordem de grandeza dos tempos registrados em cada processo.

A comparação considera os 60 minutos efetivos de elaboração das regras e mantém separados o esforço humano, o ajuste computacional final e o processamento associado à seleção dos modelos.

Os tempos de seleção foram recuperados dos Notebooks 04 e 05, executados em sessões anteriores do Google Colab, com instrumentação por `tracemalloc`. Portanto, não foram medidos nas mesmas condições da sessão atual e são utilizados principalmente para comparar ordens de grandeza, não pequenas diferenças de desempenho.

In [ ]:
# Recuperar tempos das etapas de desenvolvimento e seleção
tempos_arvores = pd.read_csv(
    pasta_tabelas / "arvores_treinamento.csv"
)

tempos_ensembles = pd.read_csv(
    pasta_tabelas / "ensembles_complexidade.csv"
)

registros_selecao = []

# Árvores: etapa inclui versão irrestrita e GridSearchCV
for _, linha in tempos_arvores.iterrows():

    tarefa = (
        "Classificação"
        if "classificação" in linha["Modelo"].lower()
        else "Regressão"
    )

    registros_selecao.append({
        "Tarefa": tarefa,
        "Modelo": "Árvore de Decisão",
        "Tempo etapa seleção (s)": linha["Tempo total (s)"],
        "Protocolo": "Árvore irrestrita + busca CV"
    })

# Ensembles: busca CV registrada no Notebook 05
for _, linha in tempos_ensembles.iterrows():

    registros_selecao.append({
        "Tarefa": linha["Tarefa"],
        "Modelo": linha["Modelo"],
        "Tempo etapa seleção (s)": linha["Tempo busca CV (s)"],
        "Protocolo": "Busca CV e ajuste"
    })

tabela_selecao = pd.DataFrame(registros_selecao)

# Juntar os tempos do ajuste final medidos no Notebook 08
tabela_custo_construcao = tabela_treinamento[
    ["Tarefa", "Modelo", "Tempo médio (s)"]
].rename(
    columns={"Tempo médio (s)": "Ajuste final médio (s)"}
).merge(
    tabela_selecao,
    on=["Tarefa", "Modelo"],
    how="left",
    validate="one_to_one"
)

assert len(tabela_custo_construcao) == 6

assert tabela_custo_construcao[
    "Tempo etapa seleção (s)"
].notna().all()

# Esforço humano registrado: 60 minutos = 3600 segundos
tempo_humano_segundos = (
    tabela_esforco_humano["Tempo efetivo (min)"].iloc[0]
    * 60
)

tabela_custo_construcao[
    "Razão esforço humano / etapa computacional"
] = (
    tempo_humano_segundos
    / tabela_custo_construcao["Tempo etapa seleção (s)"]
)

tabela_custo_construcao.to_csv(
    pasta_tabelas / "eficiencia_custo_construcao.csv",
    index=False,
    float_format="%.16g"
)

print("CUSTO DE CONSTRUÇÃO E SELEÇÃO")
print(f"Esforço humano do especialista: {tempo_humano_segundos:.0f} s")

display(
    tabela_custo_construcao.style.format({
        "Ajuste final médio (s)": "{:.4f}",
        "Tempo etapa seleção (s)": "{:.2f}",
        "Razão esforço humano / etapa computacional": "{:.1f} vezes"
    })
)

print("\nComparação salva com sucesso!")

CUSTO DE CONSTRUÇÃO E SELEÇÃO
Esforço humano do especialista: 3600 s


,Tarefa,Modelo,Ajuste final médio (s),Tempo etapa seleção (s),Protocolo,Razão esforço humano / etapa computacional
0,Classificação,Árvore de Decisão,0.1082,71.11,Árvore irrestrita + busca CV,50.6 vezes
1,Classificação,Random Forest,1.3582,53.96,Busca CV e ajuste,66.7 vezes
2,Classificação,Gradient Boosting,0.8010,30.79,Busca CV e ajuste,116.9 vezes
3,Regressão,Árvore de Decisão,0.0667,20.41,Árvore irrestrita + busca CV,176.4 vezes
4,Regressão,Random Forest,2.6294,42.71,Busca CV e ajuste,84.3 vezes
5,Regressão,Gradient Boosting,1.3116,25.29,Busca CV e ajuste,142.3 vezes



Comparação salva com sucesso!


### 3.3 Interpretação do custo de construção

Os tempos de desenvolvimento computacional foram superiores aos tempos de um único ajuste final, pois incorporaram a avaliação de diferentes configurações e a validação cruzada.

O esforço humano de elaboração do Sistema Especialista totalizou 3.600 segundos, enquanto as etapas computacionais de treinamento e seleção dos modelos ocorreram em períodos consideravelmente menores, conforme a tabela anterior.

Essa comparação evidencia diferenças de ordem de grandeza, mas não representa uma medida completa do custo de desenvolvimento. O treinamento computacional não inclui todo o esforço humano de programação, experimentação e interpretação dos resultados, assim como os 60 minutos registrados para o Sistema Especialista não incluem necessariamente toda sua validação e manutenção.

Portanto, os tempos são apresentados como indicadores de atividades distintas, e não como custos diretamente substituíveis.

## 4. Tempo de inferência dos modelos

A eficiência de inferência é avaliada nos quatro classificadores, incluindo o Sistema Especialista, e nos três modelos de regressão.

Cada modelo recebe os mesmos 4.000 registros de teste. São realizadas 30 repetições cronometradas, precedidas por uma execução de aquecimento.

A medição compreende o pré-processamento dos atributos e a geração das previsões, representando o funcionamento completo do modelo sobre dados de entrada ainda não transformados.

Para cada modelo, são calculados o tempo médio de inferência, o desvio padrão, o tempo médio por amostra, a vazão em amostras por segundo e o pico de alocação de memória Python.

A comparação é realizada na mesma sessão computacional, usando `src/profiling.py`. Os tempos podem sofrer influência da instrumentação, das condições de execução e dos mecanismos internos de cada algoritmo. O pico de alocação Python não corresponde ao consumo total de RAM.

**Diferença em relação aos Notebooks 03, 04 e 05:** nesses experimentos, a inferência foi cronometrada após o pré-processamento, utilizando registros previamente transformados. Neste notebook, a medição inclui tanto o pré-processamento quanto a geração das previsões, a partir dos 4.000 registros brutos do teste.

Assim, o protocolo atual compara o tempo de resposta dos sete sistemas completos já carregados em memória. Os tempos deste notebook não devem ser comparados diretamente com os benchmarks anteriores, pois foram utilizados protocolos de medição diferentes e as condições de execução também podem variar.

Essa mudança explica por que os tempos das Árvores de Decisão, por exemplo, são maiores neste notebook: a etapa de preparação dos dados passou a integrar a cronometragem.

In [ ]:
from src.expert_system import (
    BaseConhecimentoClassificacao,
    MotorInferenciaClassificacao
)

# -------- PREPARAR O SISTEMA ESPECIALISTA --------

# Ajustar o pré-processamento somente no treinamento
X_train_se, X_test_se, preprocessador_se = (
    pipeline_data.preparar_dados_sistema_especialista(
        X_train, X_test
    )
)

# Carregar regras e calcular frequências no treinamento
base_se = BaseConhecimentoClassificacao(
    PROJETO / "rules/regras_classificacao.txt"
)

base_se.analisar_no_treinamento(
    X_train_se, y_train_c
)

motor_se = MotorInferenciaClassificacao(base_se)

# Recebe dados brutos, aplica o pré-processamento e infere
def prever_sistema_especialista(X_bruto):

    dados = preprocessador_se.transform(X_bruto)

    dados_preparados = pd.DataFrame(
        dados,
        columns=preprocessador_se.get_feature_names_out(),
        index=X_bruto.index
    )

    return motor_se.predict(dados_preparados)


# -------- ORGANIZAR OS SETE MODELOS --------

experimentos_inferencia = [
    (
        "Classificação",
        "Sistema Especialista",
        prever_sistema_especialista
    )
]

experimentos_inferencia += [
    ("Classificação", nome, modelo.predict)
    for nome, modelo in modelos_classificacao.items()
]

experimentos_inferencia += [
    ("Regressão", nome, modelo.predict)
    for nome, modelo in modelos_regressao.items()
]

assert len(experimentos_inferencia) == 7

# -------- MEDIR INFERÊNCIA --------

N_REPETICOES_INFERENCIA = 30

registros_inferencia = []
detalhes_inferencia = []

for tarefa, nome, funcao_predicao in experimentos_inferencia:

    print(f"Medindo: {nome} — {tarefa}")

    resultado = profiling.medir_inferencia(
        funcao_predicao,
        X_test,
        n_rep=N_REPETICOES_INFERENCIA,
        n_aquecimento=1,
        retornar_detalhes=True
    )

    assert resultado["n_amostras"] == 4000
    assert resultado["n_repeticoes"] == 30
    assert len(resultado["detalhes"]) == 30

    registros_inferencia.append({
        "Tarefa": tarefa,
        "Modelo": nome,
        "Repetições": resultado["n_repeticoes"],
        "Amostras": resultado["n_amostras"],
        "Tempo médio (s)": resultado["tempo_medio_s"],
        "Desvio padrão (s)": resultado["tempo_desvio_s"],
        "Tempo por amostra (ms)": (
            resultado["tempo_por_amostra_ms"]
        ),
        "Vazão (amostras/s)": (
            resultado["vazao_amostras_s"]
        ),
        "Pico alocação Python (KB)": (
            resultado["pico_memoria_kb"]
        )
    })

    for detalhe in resultado["detalhes"]:
        detalhes_inferencia.append({
            "Tarefa": tarefa,
            "Modelo": nome,
            "Repetição": detalhe["repeticao"],
            "Tempo (s)": detalhe["tempo_s"],
            "Pico alocação Python (KB)": (
                detalhe["pico_memoria_python_kb"]
            ),
            "Vazão (amostras/s)": (
                detalhe["vazao_amostras_s"]
            )
        })

# -------- ORGANIZAR E VALIDAR --------

tabela_inferencia = pd.DataFrame(registros_inferencia)

tabela_inferencia_detalhes = pd.DataFrame(
    detalhes_inferencia
)

assert len(tabela_inferencia) == 7
assert len(tabela_inferencia_detalhes) == 210

assert np.isfinite(
    tabela_inferencia.select_dtypes(
        include="number"
    ).to_numpy()
).all()

# -------- SALVAR OS RESULTADOS --------

tabela_inferencia.to_csv(
    pasta_tabelas / "eficiencia_inferencia.csv",
    index=False,
    float_format="%.16g"
)

tabela_inferencia_detalhes.to_csv(
    pasta_logs / "medicoes_inferencia_final.csv",
    index=False,
    float_format="%.16g"
)

# -------- EXIBIR RESULTADOS --------

print("\nEFICIÊNCIA — INFERÊNCIA DOS SETE MODELOS")

display(
    tabela_inferencia.style.format({
        "Tempo médio (s)": "{:.6f}",
        "Desvio padrão (s)": "{:.6f}",
        "Tempo por amostra (ms)": "{:.6f}",
        "Vazão (amostras/s)": "{:.2f}",
        "Pico alocação Python (KB)": "{:.2f}"
    })
)

print("\nMedições de inferência salvas com sucesso!")

Medindo: Sistema Especialista — Classificação
Medindo: Árvore de Decisão — Classificação
Medindo: Random Forest — Classificação
Medindo: Gradient Boosting — Classificação
Medindo: Árvore de Decisão — Regressão
Medindo: Random Forest — Regressão
Medindo: Gradient Boosting — Regressão

EFICIÊNCIA — INFERÊNCIA DOS SETE MODELOS


,Tarefa,Modelo,Repetições,Amostras,Tempo médio (s),Desvio padrão (s),Tempo por amostra (ms),Vazão (amostras/s),Pico alocação Python (KB)
0,Classificação,Sistema Especialista,30,4000,0.053798,0.021846,0.013449,74352.89,544.09
1,Classificação,Árvore de Decisão,30,4000,0.042169,0.011454,0.010542,94857.17,545.43
2,Classificação,Random Forest,30,4000,0.051228,0.005837,0.012807,78082.51,544.90
3,Classificação,Gradient Boosting,30,4000,0.026656,0.002266,0.006664,150062.69,547.05
4,Regressão,Árvore de Decisão,30,4000,0.021480,0.002537,0.005370,186218.83,547.00
5,Regressão,Random Forest,30,4000,0.114258,0.026142,0.028564,35008.63,544.90
6,Regressão,Gradient Boosting,30,4000,0.044151,0.003303,0.011038,90598.69,545.61



Medições de inferência salvas com sucesso!


### 4.1 Interpretação dos tempos de inferência

Os tempos médios de inferência foram obtidos a partir de 30 repetições sobre lotes de 4.000 registros, incluindo o pré-processamento.

Na classificação, os quatro modelos apresentaram tempos da mesma ordem de grandeza, com variações no posicionamento entre as três rodadas de medição. A Random Forest apresentou o maior tempo em duas rodadas, enquanto o Sistema Especialista foi o mais lento na terceira.

Na regressão, a Random Forest apresentou consistentemente os maiores tempos de inferência. A Árvore de Decisão e o Gradient Boosting registraram tempos menores, alternando suas posições entre rodadas.

Essas diferenças mostram que não é possível estabelecer uma superioridade consistente de velocidade entre a Árvore de Decisão e o Gradient Boosting. A proximidade dos resultados, entretanto, não comprova equivalência estatística.

Os desvios padrão também variaram entre os modelos. Diferenças pequenas de tempo devem ser interpretadas com cautela, especialmente em ambientes computacionais compartilhados.

O tempo médio por amostra foi calculado a partir do processamento completo do lote, não representando diretamente a latência de uma requisição individual.

Os picos de alocação Python foram semelhantes entre os modelos. Como o `tracemalloc` não captura todas as alocações realizadas em código nativo, esses valores não permitem comparar adequadamente o consumo total de RAM.

As medições incluem o pré-processamento e a geração das previsões, mas não o carregamento inicial dos modelos.



### 4.2 Variação entre execuções

Para verificar a estabilidade dos resultados, foram comparadas três rodadas completas do benchmark de inferência.

Os valores apresentados representam o tempo médio de processamento de 4.000 registros, em segundos.

| Tarefa | Modelo | Execução 1 (s) | Execução 2 (s) | Execução 3 (s) |
|---|---|---:|---:|---:|
| Classificação | Sistema Especialista | 0,0461 | 0,0579 | 0,0538 |
| Classificação | Árvore de Decisão | 0,0220 | 0,0347 | 0,0422 |
| Classificação | Random Forest | 0,0783 | 0,0769 | 0,0512 |
| Classificação | Gradient Boosting | 0,0413 | 0,0309 | 0,0267 |
| Regressão | Árvore de Decisão | 0,0329 | 0,0209 | 0,0215 |
| Regressão | Random Forest | 0,1104 | 0,0959 | 0,1143 |
| Regressão | Gradient Boosting | 0,0274 | 0,0287 | 0,0442 |

**Origem dos dados:** As execuções 1 e 2 foram recuperadas de saídas registradas anteriormente. A execução 3 corresponde aos resultados finais preservados nos arquivos CSV e nas tabelas deste notebook.

Na classificação, a Random Forest apresentou o maior tempo em duas das três execuções. Na terceira, o Sistema Especialista foi o mais lento. A posição relativa dos classificadores variou entre as rodadas.

Na regressão, a Random Forest manteve o maior tempo nas três execuções, enquanto a Árvore de Decisão e o Gradient Boosting alternaram suas posições.

Assim, os resultados mostram maior estabilidade na identificação da Random Forest como o regressor mais lento do que na classificação dos modelos por velocidade. As diferenças entre Árvore de Decisão e Gradient Boosting não permitem identificar uma vantagem consistente de um sobre o outro.

As três rodadas representam medições realizadas em momentos diferentes, mas não devem ser interpretadas automaticamente como experimentos independentes em máquinas distintas.

## 5. Tamanho em disco e complexidade dos modelos

A complexidade dos modelos de aprendizado de máquina é caracterizada pelo número de árvores, total de nós, total de folhas e profundidade máxima das árvores.

O tamanho em disco corresponde aos arquivos `.joblib` dos Pipelines completos, incluindo o pré-processamento e o estimador treinado.

Para o Sistema Especialista, são apresentados o número de regras e o tamanho do motor de inferência serializado, que inclui as regras e suas estatísticas. O tamanho combinado dos arquivos de código-fonte e regras é informado separadamente. A comparação possui limitações, pois o motor serializado não inclui o pré-processador externo, enquanto os modelos de aprendizado de máquina são armazenados como Pipelines completos.

Esses indicadores complementam as medições de tempo e memória e ajudam a discutir os requisitos de armazenamento e implantação dos modelos.

In [ ]:
import numpy as np
import pandas as pd

# Arquivos dos seis Pipelines já treinados
arquivos_modelos = {
    ("Classificação", "Árvore de Decisão"):
        "decision_tree_clf.joblib",
    ("Classificação", "Random Forest"):
        "random_forest_clf.joblib",
    ("Classificação", "Gradient Boosting"):
        "gradient_boosting_clf.joblib",
    ("Regressão", "Árvore de Decisão"):
        "decision_tree_reg.joblib",
    ("Regressão", "Random Forest"):
        "random_forest_reg.joblib",
    ("Regressão", "Gradient Boosting"):
        "gradient_boosting_reg.joblib"
}

registros_complexidade = []

# -------- SISTEMA ESPECIALISTA --------

arquivo_regras = PROJETO / "rules/regras_classificacao.txt"
arquivo_codigo = PROJETO / "src/expert_system.py"

from tempfile import TemporaryDirectory

# Tamanho do motor serializado, incluindo regras e estatísticas
with TemporaryDirectory() as pasta_temporaria:
    caminho_motor = (
        Path(pasta_temporaria) / "sistema_especialista.joblib"
    )

    tamanho_especialista_kb = (
        profiling.obter_tamanho_modelo_disco(
            motor_se,
            caminho_temporario=str(caminho_motor)
        )
    )

# Medida complementar: arquivos de código e regras
tamanho_codigo_regras_kb = (
    arquivo_regras.stat().st_size
    + arquivo_codigo.stat().st_size
) / 1024

registros_complexidade.append({
    "Tarefa": "Classificação",
    "Modelo": "Sistema Especialista",
    "Tamanho (KB)": tamanho_especialista_kb,
    "Número de regras": len(base_se.regras),
    "Número de árvores": 0,
    "Total de nós": 0,
    "Total de folhas": 0,
    "Profundidade máxima": 0
})

# -------- MODELOS DE APRENDIZADO DE MÁQUINA --------

for tarefa, modelos in [
    ("Classificação", modelos_classificacao),
    ("Regressão", modelos_regressao)
]:

    for nome, pipeline in modelos.items():

        # Estimador final do Pipeline
        estimador = pipeline.steps[-1][1]

        # Árvore única ou conjunto de árvores
        if hasattr(estimador, "tree_"):
            arvores = [estimador]

        elif hasattr(estimador, "estimators_"):
            arvores = np.asarray(
                estimador.estimators_,
                dtype=object
            ).ravel().tolist()

        else:
            raise TypeError(
                f"Estrutura não reconhecida: {nome}"
            )

        assert len(arvores) >= 1
        assert all(
            hasattr(arvore, "tree_")
            for arvore in arvores
        )

        numero_arvores = len(arvores)

        total_nos = sum(
            arvore.tree_.node_count
            for arvore in arvores
        )

        total_folhas = sum(
            arvore.tree_.n_leaves
            for arvore in arvores
        )

        profundidade_maxima = max(
            arvore.tree_.max_depth
            for arvore in arvores
        )

        caminho = PROJETO / "models" / (
            arquivos_modelos[(tarefa, nome)]
        )

        tamanho_kb = caminho.stat().st_size / 1024

        registros_complexidade.append({
            "Tarefa": tarefa,
            "Modelo": nome,
            "Tamanho (KB)": tamanho_kb,
            "Número de regras": 0,
            "Número de árvores": numero_arvores,
            "Total de nós": total_nos,
            "Total de folhas": total_folhas,
            "Profundidade máxima": profundidade_maxima
        })

# -------- CONSOLIDAR --------

tabela_complexidade = pd.DataFrame(
    registros_complexidade
)

assert len(tabela_complexidade) == 7
assert (
    tabela_complexidade["Tamanho (KB)"] > 0
).all()

tabela_complexidade.to_csv(
    pasta_tabelas / "eficiencia_complexidade.csv",
    index=False,
    float_format="%.16g"
)

print("TAMANHO EM DISCO E COMPLEXIDADE")
display(
    tabela_complexidade.round(2)
)

print(
    "\nSistema Especialista — código e regras: "
    f"{tamanho_codigo_regras_kb:.2f} KB"
)

TAMANHO EM DISCO E COMPLEXIDADE


,Tarefa,Modelo,Tamanho (KB),Número de regras,Número de árvores,Total de nós,Total de folhas,Profundidade máxima
0,Classificação,Sistema Especialista,3.89,11,0,0,0,0
1,Classificação,Árvore de Decisão,6.14,0,1,21,11,4
2,Classificação,Random Forest,487.60,0,50,5922,2986,6
3,Classificação,Gradient Boosting,80.61,0,50,750,400,3
4,Regressão,Árvore de Decisão,8.79,0,1,63,32,5
5,Regressão,Random Forest,888.17,0,100,12134,6117,6
6,Regressão,Gradient Boosting,88.37,0,100,700,400,2



Sistema Especialista — código e regras: 9.17 KB


### 5.1 Interpretação do tamanho e da complexidade

As Árvores de Decisão apresentaram os menores tamanhos serializados entre os modelos de aprendizado de máquina, enquanto a Random Forest apresentou os maiores tamanhos e quantidades de nós.

O Gradient Boosting apresentou arquivos menores que os da Random Forest, apesar de utilizar a mesma quantidade de estimadores em cada tarefa. Isso demonstra que a complexidade estrutural depende também da profundidade e do número de nós das árvores, e não apenas da quantidade de estimadores.

O Sistema Especialista possui 11 regras. Seu tamanho serializado corresponde ao motor de inferência, incluindo regras e estatísticas aprendidas no treinamento. O tamanho dos arquivos de código-fonte e regras é apresentado separadamente como informação complementar.

A comparação de tamanho ainda possui uma limitação: os arquivos dos modelos de aprendizado de máquina incluem o Pipeline de pré-processamento, enquanto o motor serializado do Sistema Especialista não inclui seu pré-processador externo. Além disso, nenhum desses valores representa o consumo total de memória durante a execução.

Os resultados indicam que o tamanho em disco e a complexidade estrutural são indicadores úteis para analisar requisitos de armazenamento, mas precisam ser considerados juntamente com os tempos de inferência e os recursos efetivamente disponíveis no ambiente de implantação.

## 6. Estimativa de energia e emissão de CO₂

O consumo energético e as emissões de CO₂ são estimados a partir dos tempos médios de treinamento e inferência obtidos nas medições anteriores.

Foi adotada uma **potência hipotética constante de 45 W**, definida como premissa do projeto para representar uma ordem de grandeza plausível da potência de um processador. Esse valor não foi medido no Google Colab, não corresponde ao TDP verificado da máquina utilizada e não foi fornecido pelo enunciado.

O fator de emissão de **45,8 gCO₂/kWh** foi fornecido pelo professor na atividade.

A energia é estimada pela multiplicação da potência assumida pelo tempo de execução, com conversão para kWh. A emissão estimada corresponde à energia multiplicada pelo fator de emissão.

Para os modelos de aprendizado de máquina, o total considera um treinamento médio com hiperparâmetros fixos e uma inferência média sobre 4.000 registros. Para o Sistema Especialista, considera-se somente a inferência computacional, mantendo o esforço humano de elaboração das regras em análise separada.

Esses valores não representam consumo elétrico medido diretamente. A estimativa desconsidera variações reais da potência, consumo de outros componentes do sistema, carregamento inicial dos modelos, infraestrutura do Colab e custos computacionais anteriores de seleção de hiperparâmetros.

In [ ]:
import numpy as np
import pandas as pd

# -------- ESTIMAR ENERGIA DO TREINAMENTO --------

registros_energia_treino = []

for _, linha in tabela_treinamento.iterrows():

    estimativa = profiling.estimar_energia_carbono(
        tempo_segundos=linha["Tempo médio (s)"],
        potencia_watts=profiling.POTENCIA_PADRAO_WATTS
    )

    registros_energia_treino.append({
        "Tarefa": linha["Tarefa"],
        "Modelo": linha["Modelo"],
        "Tempo treino (s)": linha["Tempo médio (s)"],
        "Energia treino (kWh)": estimativa["energia_kwh"],
        "CO2 treino (g)": estimativa["emissao_co2_g"]
    })

energia_treino = pd.DataFrame(
    registros_energia_treino
)

# -------- ESTIMAR ENERGIA DA INFERÊNCIA --------

registros_energia_inferencia = []

for _, linha in tabela_inferencia.iterrows():

    estimativa = profiling.estimar_energia_carbono(
        tempo_segundos=linha["Tempo médio (s)"],
        potencia_watts=profiling.POTENCIA_PADRAO_WATTS
    )

    registros_energia_inferencia.append({
        "Tarefa": linha["Tarefa"],
        "Modelo": linha["Modelo"],
        "Tempo inferência (s)": linha["Tempo médio (s)"],
        "Energia inferência (kWh)": estimativa["energia_kwh"],
        "CO2 inferência (g)": estimativa["emissao_co2_g"]
    })

energia_inferencia = pd.DataFrame(
    registros_energia_inferencia
)

# -------- CONSOLIDAR AS ESTIMATIVAS --------

tabela_energia = energia_inferencia.merge(
    energia_treino,
    on=["Tarefa", "Modelo"],
    how="left",
    validate="one_to_one"
)

# Sistema Especialista não possui treinamento computacional
mascara_se = (
    tabela_energia["Modelo"] == "Sistema Especialista"
)

assert tabela_energia.loc[
    mascara_se, "Energia treino (kWh)"
].isna().all()

# Total: um treinamento médio + uma inferência média
# Para o Sistema Especialista, apenas inferência
tabela_energia["Energia total estimada (kWh)"] = (
    tabela_energia["Energia treino (kWh)"].fillna(0)
    + tabela_energia["Energia inferência (kWh)"]
)

tabela_energia["CO2 total estimado (g)"] = (
    tabela_energia["CO2 treino (g)"].fillna(0)
    + tabela_energia["CO2 inferência (g)"]
)

# -------- VERIFICAÇÕES --------

assert len(tabela_energia) == 7

assert tabela_energia[
    ["Energia total estimada (kWh)", "CO2 total estimado (g)"]
].notna().all().all()

assert (
    tabela_energia["Energia total estimada (kWh)"] > 0
).all()

# -------- SALVAR VALORES COMPLETOS --------

tabela_energia.to_csv(
    pasta_tabelas / "eficiencia_energia_co2.csv",
    index=False,
    float_format="%.16g"
)

# -------- EXIBIR RESULTADOS --------

print("ESTIMATIVAS DE ENERGIA E CO₂")
print("Potência assumida: 45 W")
print("Fator de emissão: 45,8 gCO₂/kWh")
print("Inferência: lote de 4.000 registros")
print(
    "Total: um treinamento médio + uma inferência média;"
    " para o Sistema Especialista, somente inferência."
)

display(
    tabela_energia.style.format({
        "Tempo treino (s)": "{:.6f}",
        "Tempo inferência (s)": "{:.6f}",
        "Energia treino (kWh)": "{:.6e}",
        "CO2 treino (g)": "{:.6e}",
        "Energia inferência (kWh)": "{:.6e}",
        "CO2 inferência (g)": "{:.6e}",
        "Energia total estimada (kWh)": "{:.6e}",
        "CO2 total estimado (g)": "{:.6e}"
    }, na_rep="Não se aplica")
)

print("\nEstimativas salvas com sucesso!")

ESTIMATIVAS DE ENERGIA E CO₂
Potência assumida: 45 W
Fator de emissão: 45,8 gCO₂/kWh
Inferência: lote de 4.000 registros
Total: um treinamento médio + uma inferência média; para o Sistema Especialista, somente inferência.


,Tarefa,Modelo,Tempo inferência (s),Energia inferência (kWh),CO2 inferência (g),Tempo treino (s),Energia treino (kWh),CO2 treino (g),Energia total estimada (kWh),CO2 total estimado (g)
0,Classificação,Sistema Especialista,0.053798,6.724688e-07,3.079907e-05,Não se aplica,Não se aplica,Não se aplica,6.724688e-07,3.079907e-05
1,Classificação,Árvore de Decisão,0.042169,5.271083e-07,2.414156e-05,0.108197,1.352464e-06,6.194286e-05,1.879573e-06,8.608442e-05
2,Classificação,Random Forest,0.051228,6.403483e-07,2.932795e-05,1.358229,1.697787e-05,7.775864e-04,1.761822e-05,8.069143e-04
3,Classificação,Gradient Boosting,0.026656,3.331941e-07,1.526029e-05,0.801049,1.001311e-05,4.586005e-04,1.034631e-05,4.738608e-04
4,Regressão,Árvore de Decisão,0.021480,2.685013e-07,1.229736e-05,0.066695,8.336914e-07,3.818307e-05,1.102193e-06,5.048043e-05
5,Regressão,Random Forest,0.114258,1.428219e-06,6.541244e-05,2.629402,3.286752e-05,1.505333e-03,3.429574e-05,1.570745e-03
6,Regressão,Gradient Boosting,0.044151,5.518844e-07,2.527630e-05,1.311621,1.639526e-05,7.509031e-04,1.694715e-05,7.761794e-04



Estimativas salvas com sucesso!


### 6.1 Interpretação das estimativas energéticas

As estimativas energéticas acompanham diretamente os tempos de processamento, pois foi utilizada uma potência constante de 45 W para todos os modelos.

Entre os modelos de aprendizado de máquina, as Árvores de Decisão apresentaram os menores consumos totais estimados nas duas tarefas, considerando um treinamento médio e uma inferência média. A Random Forest apresentou os maiores valores.

Na inferência isolada, os menores consumos acompanharam os menores tempos observados na tabela de inferência. As emissões estimadas seguiram a mesma proporção, em função do fator fixo de 45,8 gCO₂/kWh.

Os valores são pequenos devido à curta duração das execuções. Entretanto, representam estimativas teóricas baseadas em potência assumida, não medições elétricas diretas.

O total apresentado não contempla todo o processo de desenvolvimento dos modelos, especialmente a busca de hiperparâmetros, a validação cruzada e as demais execuções experimentais.

Para o Sistema Especialista, o cálculo considera apenas a inferência. Os 60 minutos de esforço humano de elaboração das regras permanecem registrados separadamente e não são convertidos em energia computacional.

A comparação energética deve ser interpretada juntamente com o desempenho preditivo, a complexidade dos modelos e as limitações do ambiente utilizado.

## 7. Discussão sobre implantação em dispositivos edge e servidores

A escolha de um modelo para implantação depende do equilíbrio entre desempenho preditivo, tempo de inferência, armazenamento, memória disponível e interpretabilidade.

### 7.1 Implantação em dispositivos edge

As Árvores de Decisão apresentam vantagens para dispositivos com recursos limitados devido à simplicidade estrutural e ao reduzido tamanho dos arquivos serializados.

O Sistema Especialista também possui estrutura compacta e interpretável, utilizando 11 regras. Entretanto, seu funcionamento exige o motor de inferência e as dependências necessárias para o pré-processamento e a execução.

Os ensembles apresentam maior complexidade estrutural. Entre eles, o Gradient Boosting utilizou arquivos significativamente menores que os da Random Forest nas duas tarefas.

A Árvore de Decisão e o Gradient Boosting apresentaram tempos de inferência próximos, com alternância de posição entre execuções. A Árvore de Decisão destacou-se pela simplicidade e pelo menor tamanho serializado, enquanto o Gradient Boosting apresentou desempenho preditivo competitivo com armazenamento inferior ao da Random Forest. A escolha para dispositivos edge deve considerar essas características em conjunto, sem estabelecer superioridade definitiva de velocidade a partir de pequenas diferenças de tempo.

Não foram realizados experimentos em dispositivos edge reais. Além disso, os picos de alocação Python não permitiram distinguir adequadamente o consumo de RAM dos modelos. Portanto, os resultados indicam alternativas promissoras, mas não comprovam a viabilidade de implantação em equipamentos específicos.

### 7.2 Implantação em servidores

Em servidores, a maior disponibilidade de recursos permite considerar modelos mais complexos, desde que o ganho preditivo justifique seus custos computacionais.

Na classificação, o Gradient Boosting apresentou os maiores valores de ROC-AUC e Average Precision no Notebook 06, enquanto o Sistema Especialista alcançou o maior F1-score e Recall no teste final. Em relação à eficiência, os tempos de inferência variaram entre as três rodadas, sem uma ordem consistentemente mantida entre os classificadores. A Random Forest apresentou o maior tempo em duas das três execuções.

Na regressão, o Gradient Boosting apresentou o menor RMSE no teste final. Em termos computacionais, seus tempos de inferência foram próximos aos da Árvore de Decisão, sem vantagem consistente de um modelo sobre o outro entre as execuções. A Random Forest apresentou maior tempo de inferência e armazenamento, sem melhoria preditiva expressiva que justificasse claramente esse custo adicional neste experimento.

### 7.3 Limitações das recomendações

As medições ocorreram no Google Colab, utilizando processamento em lote e modelos previamente carregados em memória.

Não foram avaliados latência de rede, concorrência entre usuários, inicialização dos serviços, execução em dispositivos embarcados, consumo real de RAM ou consumo elétrico medido diretamente.

Os resultados energéticos utilizam uma potência hipotética constante de 45 W. Portanto, não representam medições físicas do hardware utilizado.

Além disso, o estudo não constitui validação clínica. As conclusões referem-se à comparação experimental dos modelos e não à sua utilização em decisões médicas reais.

## 8. Síntese da eficiência computacional

A análise de eficiência computacional comparou os sete modelos do projeto, considerando tempo de treinamento, inferência, vazão, alocação de memória Python, armazenamento, complexidade estrutural, energia e emissões estimadas de CO₂.

**No treinamento**, as Árvores de Decisão apresentaram os menores tempos médios nas duas tarefas, enquanto a Random Forest apresentou os maiores. As medições também evidenciaram variação entre repetições, principalmente nos ensembles. Os tempos de seleção de hiperparâmetros foram apresentados separadamente, permitindo distinguir o ajuste final do processo de desenvolvimento computacional.

**Na inferência**, os classificadores apresentaram variação na ordem de velocidade entre as três rodadas. A Random Forest foi o classificador mais lento em duas delas, enquanto o Sistema Especialista ocupou essa posição na terceira. Na regressão, a Random Forest apresentou consistentemente os maiores tempos. A Árvore de Decisão e o Gradient Boosting alternaram suas posições entre execuções, sem demonstrar superioridade consistente de velocidade. Esses resultados reforçam a importância de considerar a variabilidade dos benchmarks.

A inferência do Notebook 08 incluiu o pré-processamento, diferentemente dos benchmarks dos Notebooks 03, 04 e 05. Portanto, seus tempos não devem ser comparados diretamente com os registros anteriores.

**No armazenamento e na complexidade**, as Árvores de Decisão apresentaram os menores arquivos entre os modelos de aprendizado de máquina, enquanto a Random Forest apresentou as maiores estruturas. O Gradient Boosting utilizou árvores menores e arquivos mais compactos que os da Random Forest.

O Sistema Especialista foi construído com 11 regras, mediante 60 minutos efetivos de trabalho humano registrado. Seu motor serializado apresentou tamanho reduzido, embora não inclua o pré-processamento externo. O custo humano de elaboração foi apresentado separadamente do treinamento e da seleção computacional dos demais modelos.

**Na análise energética**, as Árvores de Decisão apresentaram os menores consumos totais estimados entre os modelos de aprendizado de máquina, considerando um ajuste final médio e uma inferência média. A Random Forest apresentou os maiores consumos estimados.

A potência constante de 45 W é uma hipótese do projeto, enquanto o fator de emissão de 45,8 gCO₂/kWh foi fornecido no enunciado. Assim, os valores apresentados são estimativas, não medições elétricas diretas.

Os picos de alocação Python foram muito semelhantes entre os modelos e não forneceram uma medida suficientemente discriminativa de seu consumo real de RAM. O tamanho em disco e a complexidade estrutural ofereceram informações complementares, mas também não substituem uma avaliação de memória em ambiente de produção.

Para dispositivos edge, as Árvores de Decisão e o Sistema Especialista apresentam vantagens potenciais de simplicidade e armazenamento. Em servidores, modelos mais complexos, como Gradient Boosting e Random Forest, podem ser considerados conforme as exigências de desempenho preditivo e os recursos disponíveis.

**Conclui-se que a escolha de um modelo não deve considerar apenas a velocidade ou o consumo energético estimado.** É necessário avaliar conjuntamente qualidade das previsões, generalização, interpretabilidade, tempo de resposta e custo computacional.

As conclusões permanecem restritas ao conjunto de dados e ao ambiente experimental utilizados, sem estabelecer viabilidade comprovada de implantação ou validade clínica.

## 9. Tabela-resumo final

Esta seção consolida, em uma única tabela por tarefa, os principais resultados dos Notebooks 03 a 08: desempenho no teste, validação cruzada, generalização, eficiência computacional e custo de construção, conforme a Seção 6 do enunciado. Os valores são lidos das tabelas já salvas, sem novo treinamento ou nova medição.

In [ ]:
# Tabela-resumo final: consolida as tabelas já salvas pelos Notebooks 03 a 08.

from pathlib import Path
import pandas as pd

# Reaproveitar o ambiente da Seção 1, ou configurá-lo caso a sessão tenha sido reiniciada
if "PROJETO" not in globals():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJETO = Path("/content/drive/MyDrive/projeto-cardiovascular-ia-02")

pasta_tabelas = PROJETO / "reports" / "tables"

def ler(nome):
    return pd.read_csv(pasta_tabelas / nome)

ordem = ["Sistema Especialista", "Árvore de Decisão", "Random Forest", "Gradient Boosting"]

# Tabelas de origem
teste_c = ler("comparacao_classificacao.csv")
teste_r = ler("comparacao_regressao.csv")
cv_c = ler("generalizacao_cv_classificacao_final.csv")
cv_r = ler("generalizacao_cv_regressao_final.csv")
complexidade = ler("eficiencia_complexidade.csv")
energia = ler("eficiencia_energia_co2.csv")
construcao = ler("eficiencia_custo_construcao.csv")
esforco = ler("eficiencia_esforco_humano_regras.csv")

def descrever_estrutura(linha):
    if linha["Número de regras"] > 0:
        return f"{int(linha['Número de regras'])} regras SE–ENTÃO"
    arvores = int(linha["Número de árvores"])
    rotulo = "árvore" if arvores == 1 else "árvores"
    return (f"{arvores} {rotulo}; {int(linha['Total de folhas'])} folhas; "
            f"prof. máx. {int(linha['Profundidade máxima'])}")

def eficiencia_da_tarefa(tarefa):
    ef = complexidade[complexidade["Tarefa"] == tarefa].merge(
        energia[energia["Tarefa"] == tarefa], on=["Tarefa", "Modelo"]
    ).merge(
        construcao[construcao["Tarefa"] == tarefa], on=["Tarefa", "Modelo"], how="left"
    )
    ef["Estrutura"] = ef.apply(descrever_estrutura, axis=1)

    custo = []
    for _, linha in ef.iterrows():
        if linha["Modelo"] == "Sistema Especialista":
            minutos = esforco.loc[0, "Tempo efetivo (min)"]
            custo.append(f"Manual: {minutos:.0f} min ({int(esforco.loc[0, 'Regras elaboradas'])} regras)")
        else:
            custo.append(f"Seleção CV: {linha['Tempo etapa seleção (s)']:.1f} s")
    ef["Custo de construção"] = custo

    return ef[[
        "Modelo", "Tempo treino (s)", "Tempo inferência (s)", "Tamanho (KB)",
        "Estrutura", "CO2 total estimado (g)", "Custo de construção",
    ]].rename(columns={
        "Tempo treino (s)": "Treino final (s)",
        "Tempo inferência (s)": "Inferência 4.000 amostras (s)",
        "CO2 total estimado (g)": "CO₂ total estimado (g)",
    })

# ---------------- Classificação ----------------
resumo_c = teste_c[["Modelo", "Accuracy", "Precision", "Recall", "F1-score",
                    "ROC-AUC", "Average_Precision"]].rename(columns={
    "F1-score": "F1 teste", "Average_Precision": "Average Precision"
})
resumo_c = resumo_c.merge(
    cv_c[["Modelo", "F1 médio validação CV", "Desvio padrão CV", "Gap treino - validação CV"]]
    .rename(columns={"F1 médio validação CV": "F1 validação CV",
                     "Desvio padrão CV": "DP F1 CV",
                     "Gap treino - validação CV": "Gap F1 (treino − validação CV)"}),
    on="Modelo",
).merge(eficiencia_da_tarefa("Classificação"), on="Modelo")

# ---------------- Regressão ----------------
resumo_r = teste_r[["Modelo", "MAE", "RMSE", "R2"]].rename(columns={
    "RMSE": "RMSE teste", "R2": "R²"
})
resumo_r = resumo_r.merge(
    cv_r[["Modelo", "RMSE médio validação CV", "Desvio padrão CV", "Gap treino - validação CV"]]
    .rename(columns={"RMSE médio validação CV": "RMSE validação CV",
                     "Desvio padrão CV": "DP RMSE CV",
                     "Gap treino - validação CV": "Gap RMSE (treino − validação CV)"}),
    on="Modelo",
).merge(eficiencia_da_tarefa("Regressão"), on="Modelo")

# Diagnóstico consolidado no Notebook 07
for tabela in (resumo_c, resumo_r):
    tabela["Diagnóstico (NB07)"] = "Bem ajustado"
    tabela["Modelo"] = pd.Categorical(tabela["Modelo"], categories=ordem, ordered=True)
    tabela.sort_values("Modelo", inplace=True)
    tabela.reset_index(drop=True, inplace=True)

# O Sistema Especialista não possui etapa de treinamento computacional
resumo_c["Treino final (s)"] = resumo_c["Treino final (s)"].astype(object)
resumo_c.loc[resumo_c["Modelo"] == "Sistema Especialista", "Treino final (s)"] = "Não se aplica"

# Conferências de integridade
assert len(resumo_c) == 4 and resumo_c["F1 teste"].notna().all()
assert len(resumo_r) == 3 and resumo_r["RMSE teste"].notna().all()

# Salvar
resumo_c.to_csv(pasta_tabelas / "resumo_final_classificacao.csv", index=False)
resumo_r.to_csv(pasta_tabelas / "resumo_final_regressao.csv", index=False)

# Exibir
def formatar(tabela):
    formatos = {coluna: "{:.4f}" for coluna in tabela.select_dtypes("number").columns}
    formatos.update({c: "{:.2f}" for c in tabela.columns if c == "Tamanho (KB)"})
    formatos.update({c: "{:.2e}" for c in tabela.columns if c.startswith("CO₂")})
    formatos["Treino final (s)"] = lambda v: v if isinstance(v, str) else f"{v:.4f}"
    return tabela.style.format(formatos, na_rep="—").hide(axis="index")

print("TABELA-RESUMO FINAL — CLASSIFICAÇÃO (alvo: Cardiovascular_Disease; teste com 4.000 amostras)")
display(formatar(resumo_c))

print("\nTABELA-RESUMO FINAL — REGRESSÃO (alvo: Risk_Score; teste com 4.000 amostras)")
display(formatar(resumo_r))

print("\nObservação: o Sistema Especialista não participa da regressão, conforme orientação do professor.")
print(f"Tabelas salvas em: {pasta_tabelas}")

Mounted at /content/drive
TABELA-RESUMO FINAL — CLASSIFICAÇÃO (alvo: Cardiovascular_Disease; teste com 4.000 amostras)


Modelo,Accuracy,Precision,Recall,F1 teste,ROC-AUC,Average Precision,F1 validação CV,DP F1 CV,Gap F1 (treino − validação CV),Treino final (s),Inferência 4.000 amostras (s),Tamanho (KB),Estrutura,CO₂ total estimado (g),Custo de construção,Diagnóstico (NB07)
Sistema Especialista,0.7728,0.6331,0.6119,0.6224,0.7909,0.6213,0.6414,0.0045,0.0000,Não se aplica,0.0538,3.89,11 regras SE–ENTÃO,3.08e-05,Manual: 60 min (11 regras),Bem ajustado
Árvore de Decisão,0.7795,0.6583,0.5809,0.6172,0.7842,0.6121,0.6349,0.0057,0.0000,0.1082,0.0422,6.14,1 árvore; 11 folhas; prof. máx. 4,8.61e-05,Seleção CV: 71.1 s,Bem ajustado
Random Forest,0.7845,0.6874,0.5425,0.6064,0.8006,0.6921,0.6248,0.0061,0.0064,1.3582,0.0512,487.60,50 árvores; 2986 folhas; prof. máx. 6,8.07e-04,Seleção CV: 54.0 s,Bem ajustado
Gradient Boosting,0.7865,0.6884,0.5523,0.6129,0.8009,0.6935,0.6239,0.0033,0.0048,0.8010,0.0267,80.61,50 árvores; 400 folhas; prof. máx. 3,4.74e-04,Seleção CV: 30.8 s,Bem ajustado



TABELA-RESUMO FINAL — REGRESSÃO (alvo: Risk_Score; teste com 4.000 amostras)


Modelo,MAE,RMSE teste,R²,RMSE validação CV,DP RMSE CV,Gap RMSE (treino − validação CV),Treino final (s),Inferência 4.000 amostras (s),Tamanho (KB),Estrutura,CO₂ total estimado (g),Custo de construção,Diagnóstico (NB07)
Árvore de Decisão,2.3836,2.9707,0.4756,2.9834,0.0398,-0.0194,0.0667,0.0215,8.79,1 árvore; 32 folhas; prof. máx. 5,5.05e-05,Seleção CV: 20.4 s,Bem ajustado
Random Forest,2.3704,2.9529,0.4819,2.9703,0.0408,-0.0421,2.6294,0.1143,888.17,100 árvores; 6117 folhas; prof. máx. 6,1.57e-03,Seleção CV: 42.7 s,Bem ajustado
Gradient Boosting,2.3701,2.9526,0.4820,2.9627,0.0406,-0.0147,1.3116,0.0442,88.37,100 árvores; 400 folhas; prof. máx. 2,7.76e-04,Seleção CV: 25.3 s,Bem ajustado



Observação: o Sistema Especialista não participa da regressão, conforme orientação do professor.
Tabelas salvas em: /content/drive/MyDrive/projeto-cardiovascular-ia-02/reports/tables
